#COLAB CODE

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install --upgrade --force-reinstall torch torchvision torchaudio

In [ ]:
!pip install tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports and Data Loading

In [8]:
import json
import pandas as pd
import torch
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [9]:
PATH_COLLECTION_DATA = '/content/drive/MyDrive/CLEF_prvt/Fall 25/Data/collection_baseline.csv'
df_collection = pd.read_csv(PATH_COLLECTION_DATA)

In [10]:
PATH_QUERY_DATA = '/content/drive/MyDrive/CLEF_prvt/Fall 25/Data/queries_train_baseline.csv'
df_query = pd.read_csv(PATH_QUERY_DATA)

In [ ]:
df_collection.columns

Index(['cord_uid', 'title_processed', 'abstract_processed'], dtype='object')

In [ ]:
df_query.columns

Index(['post_id', 'cord_uid', 'tweet_text', 'tweet_text_processed'], dtype='object')

In [ ]:
df_query.columns

Index(['post_id', 'tweet_text', 'cord_uid'], dtype='object')

In [ ]:
df_query

,post_id,cord_uid,tweet_text,tweet_text_processed
0,0,htlvpvz5,Oral care in rehabilitation medicine: oral vul...,Oral care in rehabilitation medicine: oral vul...
1,1,4kfl29ul,this study isn't receiving sufficient attentio...,this study isn't receiving sufficient attentio...
2,2,jtwb17u8,"thanks, xi jinping. a reminder that this study...","thanks, xi jinping. a reminder that this study..."
3,3,0w9k8iy1,Taiwan - a population of 23 million has had ju...,Taiwan - a population of 23 million has had ju...
4,4,tiqksd69,Obtaining a diagnosis of autism in lower incom...,Obtaining a diagnosis of autism in lower incom...
...,...,...,...,...
12837,14248,9169o29b,"""evidence on covid-19 reveals a growing body o...","""evidence on covid-19 reveals a growing body o..."
12838,14249,s2bpha8l,Outdoor lighting has detrimental impacts on lo...,Outdoor lighting has detrimental impacts on lo...
12839,14250,atloc9th,"26/ and influenza virus (and other pathogens, ...","26/ and influenza virus (and other pathogens, ..."
12840,14251,t4y1ylb3,does it?'sars-cov-2-naïve vaccinees had a 13.0...,does it?'sars-cov-2-naïve vaccinees had a 13.0...


In [ ]:
title_col='title_processed'
abstract_col='abstract_processed'

In [15]:
from sentence_transformers import SentenceTransformer

# Load the model
model_name = 'Snowflake/snowflake-arctic-embed-l-v2.0'
model = SentenceTransformer(model_name)

# Define the queries and documents
queries = ['what is snowflake?', 'Where can I get the best tacos?']
documents = ['The Data Cloud!', 'Mexico City of Course!']

# Compute embeddings: use `prompt_name="query"` to encode queries!
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute cosine similarity scores
scores = model.similarity(query_embeddings, document_embeddings)

# Output the results
for query, query_scores in zip(queries, scores):
    doc_score_pairs = list(zip(documents, query_scores))
    doc_score_pairs = sorted(doc_score_pairs, key=lambda x: x[1], reverse=True)
    print("Query:", query)
    for document, score in doc_score_pairs:
        print(score, document)

Query: what is snowflake?
tensor(0.2666) The Data Cloud!
tensor(0.0663) Mexico City of Course!
Query: Where can I get the best tacos?
tensor(0.2716) Mexico City of Course!
tensor(0.1085) The Data Cloud!


In [ ]:
print("Encoding research papers...")
df_collection['text'] = df_collection[title_col] + ' ' + df_collection[abstract_col]
corpus_embeddings = model.encode(df_collection['text'].tolist(), convert_to_tensor=True, show_progress_bar=True, batch_size=64)

# Save corpus embeddings to a file
with open('/content/drive/MyDrive/CLEF_prvt/Fall 25/Data/Embeddings/snowflake_corpus_embeddings.pkl', 'wb') as f:
    pickle.dump(corpus_embeddings, f)

In [ ]:
# Process queries and get embeddings
query_embeddings = model.encode(df_query['tweet_text'].tolist(), prompt_name="query", convert_to_tensor=True, show_progress_bar=True, batch_size=64)

# Save query embeddings to a file
with open('/content/drive/MyDrive/CLEF_prvt/Fall 25/Data/Embeddings/snowflake_query_embeddings.pkl', 'wb') as f:
    pickle.dump(query_embeddings, f)

Building Dataset

In [6]:
import pickle
import torch

# Load corpus embeddings from the file
with open('/content/drive/MyDrive/CLEF_prvt/Fall 25/Data/Embeddings/snowflake_corpus_embeddings.pkl', 'rb') as f:
    loaded_corpus_embeddings = pickle.load(f)

# Sanity check
print(f"Shape of loaded corpus embeddings: {loaded_corpus_embeddings.shape}")
print(f"Type of loaded corpus embeddings: {type(loaded_corpus_embeddings)}")

Shape of loaded corpus embeddings: torch.Size([7718, 1024])
Type of loaded corpus embeddings: <class 'torch.Tensor'>


In [7]:
with open('/content/drive/MyDrive/CLEF_prvt/Fall 25/Data/Embeddings/snowflake_query_embeddings.pkl', 'rb') as f:
    loaded_query_embeddings = pickle.load(f)

print(f"Shape of loaded query embeddings: {loaded_query_embeddings.shape}")
print(f"Type of loaded query embeddings: {type(loaded_query_embeddings)}")

Shape of loaded query embeddings: torch.Size([12842, 1024])
Type of loaded query embeddings: <class 'torch.Tensor'>


In [1]:
def get_top_cord_uids(query_embedding, top_k):
    # Compute similarity scores
    scores = model.similarity(query_embedding, loaded_corpus_embeddings)

    # Get top_k indices and scores
    top_k_values, top_k_indices = torch.topk(scores, k=top_k)
    top_k_indices = top_k_indices.tolist()
    top_k_values = top_k_values.tolist()

    bi_encoder_topk = []
    bi_encoder_scores = []
    for index, score in zip(top_k_indices, top_k_values):
        bi_encoder_topk.append(df_collection.iloc[index]['cord_uid'])
        bi_encoder_scores.append(score)

    return bi_encoder_topk, bi_encoder_scores

In [30]:
from tqdm import tqdm
tqdm.pandas()

df_query[['arctic_topk', 'arctic_scores']] = df_query.progress_apply(lambda row: pd.Series(get_top_cord_uids(loaded_query_embeddings[row.name], top_k=1000)), axis=1)

100%|██████████| 12842/12842 [00:21<00:00, 610.45it/s]


In [31]:
df_query

,post_id,cord_uid,tweet_text,tweet_text_processed,arctic_topk,arctic_scores
0,0,htlvpvz5,Oral care in rehabilitation medicine: oral vul...,Oral care in rehabilitation medicine: oral vul...,"[[htlvpvz5, e9uou6rr, rfekwtdb, z5povxuj, 4r96...","[[0.7585896253585815, 0.41789665818214417, 0.3..."
1,1,4kfl29ul,this study isn't receiving sufficient attentio...,this study isn't receiving sufficient attentio...,"[[wvfw94n1, 4kfl29ul, fm8koqjd, 3o7rd8pt, 29z4...","[[0.674071729183197, 0.6489344239234924, 0.645..."
2,2,jtwb17u8,"thanks, xi jinping. a reminder that this study...","thanks, xi jinping. a reminder that this study...","[[jtwb17u8, iobpcfs5, veeavho5, 944pn0k9, fe45...","[[0.6664571762084961, 0.6404362916946411, 0.56..."
3,3,0w9k8iy1,Taiwan - a population of 23 million has had ju...,Taiwan - a population of 23 million has had ju...,"[[0w9k8iy1, 91nj68tn, i1yf8fgk, hzbtd39j, l4y7...","[[0.5664387941360474, 0.4286244809627533, 0.42..."
4,4,tiqksd69,Obtaining a diagnosis of autism in lower incom...,Obtaining a diagnosis of autism in lower incom...,"[[tiqksd69, 0u330d2u, aqbhxv1f, b0dzhsrh, nh4d...","[[0.8313989043235779, 0.39427563548088074, 0.3..."
...,...,...,...,...,...,...
12837,14248,9169o29b,"""evidence on covid-19 reveals a growing body o...","""evidence on covid-19 reveals a growing body o...","[[jgq968f6, 5yscqct1, qkjieod9, s2hp3sat, 08bw...","[[0.6891266107559204, 0.6430435180664062, 0.63..."
12838,14249,s2bpha8l,Outdoor lighting has detrimental impacts on lo...,Outdoor lighting has detrimental impacts on lo...,"[[s2bpha8l, 8a3fp7ym, 307rt03e, 1otgdo19, 9rxv...","[[0.6544826626777649, 0.5775607228279114, 0.32..."
12839,14250,atloc9th,"26/ and influenza virus (and other pathogens, ...","26/ and influenza virus (and other pathogens, ...","[[c59vfq1o, atloc9th, 7pcnf2iv, olhgu24h, svi7...","[[0.4738156497478485, 0.4728795289993286, 0.45..."
12840,14251,t4y1ylb3,does it?'sars-cov-2-naïve vaccinees had a 13.0...,does it?'sars-cov-2-naïve vaccinees had a 13.0...,"[[t4y1ylb3, 7a543f7v, 0o7rm667, zctjkag0, ws3t...","[[0.728946328163147, 0.6971069574356079, 0.659..."


In [ ]:
arctic_topk = []
for i in range(len(df_query)):
  arctic_topk.append(df_query["arctic_topk"].loc[i][0].to_list())
print(len(arctic_topk),len(arctic_topk[0]))

df_query['snowflake_topk'] = arctic_topk

12842 1000


In [ ]:
acrtic_topk_scores=[]
for i in range(len(df_query)):
  acrtic_topk_scores.append(df_query["arctic_scores"].loc[i][0])
print(len(acrtic_topk_scores),len(acrtic_topk_scores[0]))

df_query['snowflake_scores'] = acrtic_topk_scores

12842 1000


In [42]:
def calculate_mrr(row, cutoff=1000):
    relevant_doc = row['cord_uid']
    ranked_list = row['arctic_topk'][:cutoff]

    try:
        rank = ranked_list.index(relevant_doc) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0

mrr_at_5 = df_query.apply(lambda row: calculate_mrr(row, cutoff=5), axis=1).mean()
print(f"MRR @ 5: {mrr_at_5}")

mrr_at_10 = df_query.apply(lambda row: calculate_mrr(row, cutoff=10), axis=1).mean()
print(f"MRR @ 10: {mrr_at_10}")

mrr_at_1000 = df_query.apply(lambda row: calculate_mrr(row, cutoff=1000), axis=1).mean()
print(f"MRR @ 1000: {mrr_at_1000}")

MRR @ 5: 0.6597661319628303
MRR @ 10: 0.6665129671242427
MRR @ 1000: 0.6719216371447533
